# Online Retail II — Data Cleaning

This notebook cleans the raw combined Online Retail II dataset for customer analytics.

The cleaned dataset will support RFM segmentation, cohort retention analysis, churn modelling, feature engineering, and dashboard reporting.

## Setup

Import the required libraries, define project paths, and confirm the working environment.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import pandas.api.types as ptypes

# Show enough columns when inspecting the dataset.
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Use the project root, even when the notebook runs from the notebooks folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw file exists: {RAW_PATH.exists()}")
print(f"Raw path: {RAW_PATH}")

Python version: 3.12.6
pandas version: 3.0.2
Project root: c:\Users\Lenovo\Desktop\projects\retail-analytics
Raw file exists: True
Raw path: c:\Users\Lenovo\Desktop\projects\retail-analytics\data\raw\online_retail_II.xlsx


## Load raw data and create working copy

Load both Excel sheets, combine them into one raw dataset, and create a working copy for cleaning.

In [2]:
# Load both yearly sheets from the raw Excel file.
sheet_2009_2010 = pd.read_excel(RAW_PATH, sheet_name="Year 2009-2010")
sheet_2010_2011 = pd.read_excel(RAW_PATH, sheet_name="Year 2010-2011")

# Combine both sheets into one raw dataset.
df_raw = pd.concat(
    [sheet_2009_2010, sheet_2010_2011],
    ignore_index=True
)

# Validate the raw dataset against the Stage 1 row and column count.
expected_shape = (1_067_371, 8)

print(f"2009-2010 sheet shape: {sheet_2009_2010.shape}")
print(f"2010-2011 sheet shape: {sheet_2010_2011.shape}")
print(f"Combined raw shape: {df_raw.shape}")

assert df_raw.shape == expected_shape, (
    f"Unexpected raw shape: {df_raw.shape}. "
    f"Expected: {expected_shape}."
)

# Create a working copy so the raw dataset remains unchanged.
df = df_raw.copy()

# Track row counts after each major cleaning step.
row_log = [("raw", len(df))]

print("\nRaw dataset loaded successfully.")
print(f"Working copy shape: {df.shape}")
print(f"Row log: {row_log}")

2009-2010 sheet shape: (525461, 8)
2010-2011 sheet shape: (541910, 8)
Combined raw shape: (1067371, 8)

Raw dataset loaded successfully.
Working copy shape: (1067371, 8)
Row log: [('raw', 1067371)]


## Text standardisation

Standardise text fields used for matching and filtering. This step should not change the number of rows.

In [3]:
rows_before = len(df)

# Cast text columns to string dtype for consistent string operations.
df["Invoice"] = df["Invoice"].astype("string")
df["StockCode"] = df["StockCode"].astype("string")
df["Description"] = df["Description"].astype("string")

# Remove leading and trailing whitespace.
df["Invoice"] = df["Invoice"].str.strip()
df["StockCode"] = df["StockCode"].str.strip()
df["Description"] = df["Description"].str.strip()

# Collapse repeated internal whitespace in descriptions.
df["Description"] = df["Description"].str.replace(r"\s+", " ", regex=True)

# Validate that no rows were removed during text standardisation.
assert len(df) == rows_before, f"Row count changed: {rows_before} -> {len(df)}"

print("Dtypes after standardisation:")
print(df[["Invoice", "StockCode", "Description"]].dtypes)

# Spot-check one description that changed during cleanup.
mask_changed = (
    df_raw["Description"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    != df_raw["Description"].astype("string").fillna("")
)

if mask_changed.any():
    example_idx = mask_changed.idxmax()
    
    print(f"\nSpot-check row: {example_idx}")
    print(f"Before: {df_raw.loc[example_idx, 'Description']!r}")
    print(f"After:  {df.loc[example_idx, 'Description']!r}")
else:
    print("\nNo descriptions required whitespace cleanup.")

print(f"\nRows: {len(df):,} unchanged")

Dtypes after standardisation:
Invoice        string
StockCode      string
Description    string
dtype: object

Spot-check row: 2
Before: ' WHITE CHERRY LIGHTS'
After:  'WHITE CHERRY LIGHTS'

Rows: 1,067,371 unchanged


### Drop rows with missing Customer ID

Rows without a `Customer ID` cannot be used for customer-level analysis, including RFM segmentation, retention analysis, churn modelling, or customer-level feature engineering.

This filter is validated against the Stage 1 anchor of 824,364 remaining rows.

In [4]:
rows_before = len(df)

# Drop rows where Customer ID is missing.
df = df[df["Customer ID"].notna()].copy()

rows_dropped = rows_before - len(df)
row_log.append(("drop_missing_customer", len(df)))

# Validate against the Stage 1 anchor.
expected_count = 824_364
assert len(df) == expected_count, (
    f"Expected {expected_count:,} rows after filter, got {len(df):,}."
)

print(f"Rows before: {rows_before:,}")
print(f"Rows dropped: {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}")

Rows before: 1,067,371
Rows dropped: 243,007
Rows remaining: 824,364


### Drop non-standard Invoice rows

Keep only invoices that match the standard 6-digit invoice pattern.

This removes cancellation invoices, bad-debt adjustments, and any other non-standard invoice records that are not suitable for normal customer purchase analysis.

In [5]:
rows_before = len(df)

# Identify rows where Invoice does not match the 6-digit standard.
valid_invoice = df["Invoice"].str.fullmatch(r"\d{6}", na=False)
rejected = df.loc[~valid_invoice, "Invoice"]

# Classify rejected invoices for audit.
def classify_invoice(invoice: str) -> str:
    invoice = str(invoice)

    if invoice.startswith("C"):
        return "C-prefix (cancellation)"
    if invoice.startswith("A"):
        return "A-prefix (bad debt)"
    if invoice.isdigit():
        return f"all-digit, wrong length ({len(invoice)})"

    return "other"


breakdown = rejected.map(classify_invoice).value_counts()

# Keep only standard 6-digit invoices.
df = df.loc[valid_invoice].copy()

rows_dropped = rows_before - len(df)
row_log.append(("drop_non_standard_invoice", len(df)))

# Validate that all remaining invoices follow the standard pattern.
assert df["Invoice"].str.fullmatch(r"\d{6}", na=False).all(), (
    "Non-standard invoices remain after filtering."
)

print(f"Rows before: {rows_before:,}")
print(f"Rows dropped: {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}\n")

print("Breakdown of rejected invoices:")
print(breakdown)

Rows before: 824,364
Rows dropped: 18,744
Rows remaining: 805,620

Breakdown of rejected invoices:
Invoice
C-prefix (cancellation)    18744
Name: count, dtype: int64


### Drop non-positive Quantity rows

Keep only rows with a positive purchased quantity.

Most negative quantities were already removed with cancellation invoices, but this check ensures no remaining non-positive quantities enter the customer analytics dataset.

In [6]:
rows_before = len(df)

# Keep only rows with a positive purchased quantity.
df = df[df["Quantity"] > 0].copy()

rows_dropped = rows_before - len(df)
row_log.append(("drop_non_positive_quantity", len(df)))

# Validate that no non-positive quantities remain.
assert (df["Quantity"] > 0).all(), "Non-positive quantities remain after filtering."

print(f"Rows before: {rows_before:,}")
print(f"Rows dropped: {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}")

Rows before: 805,620
Rows dropped: 0
Rows remaining: 805,620


### Drop non-positive Price rows

Keep only rows with a positive unit price.

This removes zero-price and negative-price records so revenue calculations are based only on valid paid transactions.

In [7]:
rows_before = len(df)

# Keep only rows with a positive unit price.
df = df[df["Price"] > 0].copy()

rows_dropped = rows_before - len(df)
row_log.append(("drop_non_positive_price", len(df)))

# Validate that no non-positive prices remain.
assert (df["Price"] > 0).all(), "Non-positive prices remain after filtering."

print(f"Rows before: {rows_before:,}")
print(f"Rows dropped: {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}")

Rows before: 805,620
Rows dropped: 71
Rows remaining: 805,549


### Drop non-product StockCodes

Remove StockCodes identified in Stage 1 as postage, fees, adjustments, test records, bad-debt-related codes, or other non-product entries.

These records are excluded because the cleaned dataset should represent normal customer product purchases.

In [8]:
# Non-product StockCodes identified in the Stage 1 audit.
# Case-sensitive: 'M' and 'm' are distinct codes in the audit.
NON_PRODUCT_CODES = [
    "POST", "DOT", "M", "m", "D", "S", "ADJUST", "AMAZONFEE",
    "CRUK", "B", "GIFT", "PADS", "BANK CHARGES", "C2",
    "TEST001", "TEST002", "ADJUST2",
]

rows_before = len(df)

# Count rows per non-product code present at this stage.
non_product_mask = df["StockCode"].isin(NON_PRODUCT_CODES)
contribution = df.loc[non_product_mask, "StockCode"].value_counts().astype("int64")

# Apply the filter.
df = df[~non_product_mask].copy()

rows_dropped = rows_before - len(df)
row_log.append(("drop_non_product_stockcodes", len(df)))

# Validate that no non-product codes remain.
assert (
    not df["StockCode"].isin(NON_PRODUCT_CODES).any()
), "Non-product StockCodes remain after filtering."

# Validate internal consistency.
assert rows_dropped == contribution.sum(), (
    f"Mismatch: rows_dropped={rows_dropped:,} "
    f"vs contribution.sum()={contribution.sum():,}"
)

print(f"Rows before: {rows_before:,}")
print(f"Rows dropped: {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}\n")

print("Rows per non-product code present at this stage:")
print(contribution)

Rows before: 805,549
Rows dropped: 2,915
Rows remaining: 802,634

Rows per non-product code present at this stage:
StockCode
POST            1838
M                709
C2               253
BANK CHARGES      32
ADJUST            32
PADS              17
DOT               16
TEST001            9
D                  5
ADJUST2            3
TEST002            1
Name: count, dtype: int64


### Drop exact duplicate rows

Remove exact full-row duplicates after the main cleaning filters.

The Stage 1 duplicate count was measured on the raw dataset, so this step is validated by confirming that no exact duplicates remain after filtering.

In [9]:
rows_before = len(df)

# Capture one true duplicate pair before dropping.
# Sorting duplicate-flagged rows by all columns places exact duplicates together.
duplicate_mask = df.duplicated(keep=False)

if duplicate_mask.any():
    example_pair = (
        df.loc[duplicate_mask]
        .sort_values(list(df.columns))
        .head(2)
    )
else:
    example_pair = None

# Drop exact full-row duplicates.
df = df.drop_duplicates().copy()

rows_dropped = rows_before - len(df)
row_log.append(("drop_exact_duplicates", len(df)))

# Validate that no exact duplicates remain.
assert df.duplicated().sum() == 0, "Duplicates remain after drop_duplicates."

print(f"Rows before: {rows_before:,}")
print(f"Rows dropped: {rows_dropped:,}")
print(f"Rows remaining: {len(df):,}")

if example_pair is not None:
    print("\nExample duplicate rows before removal:")
    print(example_pair)

Rows before: 802,634
Rows dropped: 26,055
Rows remaining: 776,579

Example duplicate rows before removal:
    Invoice StockCode                      Description  Quantity         InvoiceDate  Price  Customer ID  \
379  489517     21491  SET OF THREE VINTAGE GIFT WRAPS         1 2009-12-01 11:34:00   1.95      16329.0   
391  489517     21491  SET OF THREE VINTAGE GIFT WRAPS         1 2009-12-01 11:34:00   1.95      16329.0   

            Country  
379  United Kingdom  
391  United Kingdom  


## Convert data types

Convert `Customer ID` to nullable integer format so it behaves like an identifier rather than a decimal number.

In [10]:
rows_before = len(df)

# Confirm Customer ID values are whole numbers before integer casting.
assert (df["Customer ID"] == df["Customer ID"].astype("int64")).all(), (
    "Customer ID contains non-integer values."
)

# Cast Customer ID to nullable integer dtype.
df["Customer ID"] = df["Customer ID"].astype("Int64")

# Validate dtype, content, and row count.
assert df["Customer ID"].dtype == "Int64", (
    f"Unexpected dtype: {df['Customer ID'].dtype}."
)
assert df["Customer ID"].notna().all(), "Customer ID contains nulls after casting."
assert len(df) == rows_before, "Row count changed during Customer ID type conversion."

# Print range for awareness; filtered rows may legitimately shift the bounds.
cust_min = df["Customer ID"].min()
cust_max = df["Customer ID"].max()

print(f"Customer ID dtype: {df['Customer ID'].dtype}")
print(f"Customer ID range: {cust_min} to {cust_max}")
print("Stage 1 raw Customer ID range: 12346 to 18287")
print(f"Unique customers: {df['Customer ID'].nunique():,}")

Customer ID dtype: Int64
Customer ID range: 12346 to 18287
Stage 1 raw Customer ID range: 12346 to 18287
Unique customers: 5,852


## Create Revenue feature

Create line-level revenue as `Quantity * Price`.

This feature is required for customer value analysis, RFM segmentation, sales KPIs, and downstream dashboarding.

In [11]:
rows_before = len(df)

# Create line-level revenue.
df["Revenue"] = df["Quantity"] * df["Price"]

# Validate revenue creation.
assert len(df) == rows_before, (
    f"Row count changed: {rows_before:,} -> {len(df):,}."
)
assert df["Revenue"].notna().all(), "Revenue contains nulls."
assert (df["Revenue"] > 0).all(), "Revenue contains non-positive values."

# Spot-check that Revenue equals Quantity multiplied by Price.
spot_check = df[["Quantity", "Price", "Revenue"]].head(3).copy()
spot_check["Quantity * Price"] = spot_check["Quantity"] * spot_check["Price"]

print(f"Revenue dtype: {df['Revenue'].dtype}")
print(f"Revenue range: {df['Revenue'].min():.2f} to {df['Revenue'].max():.2f}")

print("\nSpot-check: Revenue should equal Quantity * Price.")
print(spot_check)

Revenue dtype: float64
Revenue range: 0.06 to 168469.60

Spot-check: Revenue should equal Quantity * Price.
   Quantity  Price  Revenue  Quantity * Price
0        12   6.95     83.4              83.4
1        12   6.75     81.0              81.0
2        12   6.75     81.0              81.0


## Final validation checks

Run final checks to confirm the cleaned dataset is consistent, analysis-ready, and aligned with the cleaning rules applied above.

In [12]:
# Row-count audit trail.
log_dataframe = pd.DataFrame(row_log, columns=["step", "rows_after"])
log_dataframe["rows_dropped"] = (
    log_dataframe["rows_after"].shift(1).fillna(log_dataframe["rows_after"])
    - log_dataframe["rows_after"]
)
log_dataframe["rows_dropped"] = log_dataframe["rows_dropped"].astype(int)

# Row-count consistency: final length must match the last row_log entry.
assert len(df) == row_log[-1][1], (
    f"Final length {len(df):,} does not match last row_log entry "
    f"{row_log[-1][1]:,}"
)

# Critical-column null checks.
critical_columns = [
    "Customer ID", "Invoice", "StockCode",
    "Quantity", "Price", "Revenue", "InvoiceDate",
]

for column in critical_columns:
    assert df[column].notna().all(), f"Nulls remain in {column}."

# Invoice format.
assert df["Invoice"].str.fullmatch(r"\d{6}", na=False).all(), (
    "Non-standard invoice format remains."
)

# Positivity contracts.
assert (df["Quantity"] > 0).all(), "Non-positive Quantity remains."
assert (df["Price"] > 0).all(), "Non-positive Price remains."
assert (df["Revenue"] > 0).all(), "Non-positive Revenue remains."

# Duplicate and non-product checks.
assert df.duplicated().sum() == 0, "Exact duplicates remain."
assert not df["StockCode"].isin(NON_PRODUCT_CODES).any(), (
    "Non-product StockCodes remain."
)

# Date range: Stage 1 anchor. InvoiceDate was not used for filtering.
date_minimum = df["InvoiceDate"].min()
date_maximum = df["InvoiceDate"].max()

assert date_minimum.normalize() == pd.Timestamp("2009-12-01"), (
    f"Min date changed: {date_minimum}."
)
assert date_maximum.normalize() == pd.Timestamp("2011-12-09"), (
    f"Max date changed: {date_maximum}."
)

# Dtype contracts.
assert df["Customer ID"].dtype == "Int64", (
    f"Customer ID dtype: {df['Customer ID'].dtype}."
)
assert ptypes.is_numeric_dtype(df["Revenue"]), (
    f"Revenue is not numeric: {df['Revenue'].dtype}."
)

# Print the audit trail.
print("Filter cascade:")
print(log_dataframe.to_string(index=False))

print(f"\nFinal shape: {df.shape}")
print(f"Date range: {date_minimum} to {date_maximum}")
print(f"Unique customers: {df['Customer ID'].nunique():,}")
print(f"Unique invoices: {df['Invoice'].nunique():,}")
print(f"Unique products (StockCode): {df['StockCode'].nunique():,}")
print(f"Total revenue: {df['Revenue'].sum():,.2f}")

print("\nAll validations passed.")

Filter cascade:
                       step  rows_after  rows_dropped
                        raw     1067371             0
      drop_missing_customer      824364        243007
  drop_non_standard_invoice      805620         18744
 drop_non_positive_quantity      805620             0
    drop_non_positive_price      805549            71
drop_non_product_stockcodes      802634          2915
      drop_exact_duplicates      776579         26055

Final shape: (776579, 9)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers: 5,852
Unique invoices: 36,594
Unique products (StockCode): 4,620
Total revenue: 17,068,582.72

All validations passed.


## Save cleaned dataset

Save the cleaned customer analytics dataset to `data/processed/` as a Parquet file.

Parquet is used because it preserves data types, loads faster than CSV, and is more efficient for downstream analysis.

In [13]:
# Ensure the processed directory exists.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = PROCESSED_DIR / "online_retail_cleaned.parquet"

# Save the cleaned dataset.
df.to_parquet(cleaned_path, index=False)

# Load it back to verify the file on disk matches the in-memory dataframe.
loaded = pd.read_parquet(cleaned_path)

# Shape must match exactly.
assert loaded.shape == df.shape, (
    f"Shape mismatch on round-trip: saved {df.shape}, loaded {loaded.shape}."
)

# Value-level round-trip check.
# check_dtype=False allows harmless pandas/pyarrow storage differences on string columns.
# Targeted dtype checks below cover the analytical columns that matter.
pd.testing.assert_frame_equal(
    loaded.reset_index(drop=True),
    df.reset_index(drop=True),
    check_dtype=False,
)

# Targeted dtype contracts for analytical columns.
assert loaded["Customer ID"].dtype == "Int64", (
    f"Customer ID dtype lost on round-trip: {loaded['Customer ID'].dtype}."
)
assert ptypes.is_numeric_dtype(loaded["Revenue"]), (
    f"Revenue is not numeric on reload: {loaded['Revenue'].dtype}."
)
assert ptypes.is_datetime64_any_dtype(loaded["InvoiceDate"]), (
    f"InvoiceDate is not datetime on reload: {loaded['InvoiceDate'].dtype}."
)

# Report saved file details.
file_size_mb = cleaned_path.stat().st_size / (1024 * 1024)

print(f"Saved to: {cleaned_path}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Shape: {loaded.shape}")
print(f"Customer ID dtype on reload: {loaded['Customer ID'].dtype}")
print(f"Revenue dtype on reload: {loaded['Revenue'].dtype}")
print(f"InvoiceDate dtype on reload: {loaded['InvoiceDate'].dtype}")

Saved to: c:\Users\Lenovo\Desktop\projects\retail-analytics\data\processed\online_retail_cleaned.parquet
File size: 6.03 MB
Shape: (776579, 9)
Customer ID dtype on reload: Int64
Revenue dtype on reload: float64
InvoiceDate dtype on reload: datetime64[us]


## Stage 2 cleaning summary

The raw combined Online Retail II dataset contained 1,067,371 rows and 8 columns. After cleaning, the customer analytics dataset contains 776,579 rows and 9 columns (the added column is line-level `Revenue`).

The largest filters by row count were missing customer identifiers (243,007 rows), exact duplicates (26,055), and cancellation invoices (18,744). Smaller filters removed non-product StockCodes (2,915) and non-positive prices (71). Non-positive quantities were also removed as part of the cleaning contract; on this dataset the prior filters had already eliminated all such rows, so this filter dropped zero rows. Text standardisation (whitespace stripping on `Invoice`, `StockCode`, and `Description`, plus internal-space collapse on `Description`) ran without removing rows.

The final dataset contains only standard 6-digit invoices, positive quantities, positive prices, positive revenue, and no exact duplicates. It is saved as `data/processed/online_retail_cleaned.parquet` and is ready for RFM segmentation and cohort retention analysis.

Full cleaning logic, row-count audit trail, and validation contracts are in `notebooks/02_cleaning.ipynb`.